### Video Split
- 手動把影片自動切割成多個場景切換的片段

切一分十五秒-一分五十五

In [ ]:
# 連線到雲端硬碟
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 更新系統套件庫並安裝 FFmpeg
!apt-get update -qq
!apt-get install -y -qq ffmpeg

# 安裝 PySceneDetect 與 OpenCV
!pip install scenedetect[opencv] -q

print("環境安裝完成！")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.2/102.2 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.9/130.9 kB 10.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rasterio 1.5.0 requires click!=8.2.*,>=4.0, but you have click 8.2.1 which is incompatible.
環境安裝完成！


In [ ]:
# 直接用ffmpeg切割
# -ss 00:01:15：Start Size，指定開始的時間點（時:分:秒）。
# -to 00:02:30：指定結束的時間點。
# -c copy：直接複製影像和聲音串流，不要重新編碼。
!ffmpeg -i "/content/drive/MyDrive/Audio Description/data/demo.mp4" -ss 00:01:15 -to 00:01:55 -c copy "/content/drive/MyDrive/Audio Description/data/demo_split.mp4"

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [ ]:
# Cell 1: 安裝套件
!pip install openai opencv-python

# 步驟 2：影片檔案
video_path = "/content/demo_split.mp4"


# Cell 2: 匯入套件與設定 API Key
import cv2
import base64
from openai import OpenAI
import os

#  API Key
os.environ["OPENAI_API_KEY"] = "XXXXXXXXXXXXXX"
client = OpenAI()

# 將影片轉換為 Base64 圖片序列的函數

def process_video(video_path, seconds_per_frame=2): #  修改 1：改成每 2 秒抽一張圖
    base64Frames = []
    video = cv2.VideoCapture(video_path)
    total_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = video.get(cv2.CAP_PROP_FPS)
    frames_to_skip = int(fps * seconds_per_frame)
    curr_frame = 0

    while curr_frame < total_frames - 1:
        video.set(cv2.CAP_PROP_POS_FRAMES, curr_frame)
        success, frame = video.read()
        if not success:
            break

        #  修改 2：將圖片等比例縮小（寬度縮至 512 像素），大幅降低 Token 消耗
        height, width = frame.shape[:2]
        new_width = 512
        new_height = int((new_width / width) * height)
        frame = cv2.resize(frame, (new_width, new_height))

        _, buffer = cv2.imencode(".jpg", frame)
        base64Frames.append(base64.b64encode(buffer).decode("utf-8"))
        curr_frame += frames_to_skip

    video.release()
    return base64Frames

# 重新擷取畫面
frames = process_video("/content/demo_split.mp4")
print(f"成功擷取 {len(frames)} 張畫面！(已縮小尺寸與張數)")

成功擷取 21 張畫面！(已縮小尺寸與張數)


### 華語準則

In [ ]:

system_prompt_mandarin = """
你是一位頂尖的華語電影口述影像（Audio Description, AD）專家。你的任務是將視覺符碼轉譯為聽覺語言，協助視障者在腦海中建構影像。

請根據使用者提供的連續畫面，嚴格遵守以下由「華語 AD 核心規範」萃取出的準則來撰寫詞稿，特別針對《刺客聶隱娘》這類高語境、低對白藝術電影：

1. 【客觀性邊界與解讀權】：必須客觀、中立地描述畫面中實際呈現的內容。絕對禁止臆測或描寫角色的「內心情緒」與「心裡想法」，必須將劇情與畫面的解讀權完全交還給視障聽眾。
2. 【動作與微表情優先】：因本片對白極少，必須優先描述能「填補台詞未提及關鍵資訊」的動作與微表情（如視線流轉、握緊匕首），以非語言訊息推動劇情。
3. 【精煉動詞與文化語境】：必須優先使用生動、具象、畫面感強的動詞（如：掠、刺、凝視、斂衽）。詞彙必須考量華語文化圈的認知習慣，展現唐代武俠「冷酷、寫意」的東方美學。
4. 【嚴禁濫用時間副詞】：嚴格禁止在句子開頭無差別濫用「此時、現在、這時、接著」等時間副詞，避免破壞電影原有的靜謐節奏與文學性。
5. 【認知負荷管控】：並非資訊越多越好。必須結合時長限制，優先篩選與「劇情核心」相關的視覺元素，捨棄無關緊要的非核心細節。
6. 【字數適配】：影片長達 40 秒，請輸出約 120-150 個繁體中文字的描述稿，使用簡潔通順的短句，讓配音員有充足的時間娓娓道來。

請以 JSON 格式輸出：
{
  "visual_features": "簡述你提取到的關鍵視覺特徵與美學氛圍",
  "ad_script": "最終生成的口述影像詞稿（120-150字）"
}
"""

### 華語準則 有提示

In [ ]:
def generate_ad(system_prompt, image_frames):

    user_prompt_text = """
    這是電影《刺客聶隱娘》的一段連續畫面（按時間先後順序排列，總長約 40 秒）。

    【人類專家劇情提示】
    注意這 40 秒內的節奏變化，這是一個「從靜到動」的過程：
    1. 前半段（靜）：隱娘在黑白斑駁的樹林中靜靜埋伏、穿梭。一位男子騎著裝飾華麗的馬匹緩緩經過。
    2. 後半段（動）：隱娘突然如鬼魅般逼近發動伏擊，動作極快。
    3. 結局：騎馬的男子被一擊斃命，從馬上墜落，畫面變得劇烈搖晃且模糊。
#自己寫的任務prompt
    【你的任務】
    請結合你看到的畫面與上述專家提示，撰寫這 40 秒的口述影像。
    請利用 120-150 字的篇幅，把前半段的「壓迫感與靜謐」跟後半段的「一擊必殺」的節奏對比寫出來。明確點出男子被擊殺墜馬的結局。
    請特別注意：描述刺殺動作時，必須遵守 System 準則中的「客觀性」與「精煉動詞」，不要寫成血腥的賽事解說，保留寫意美學。

    請以 JSON 格式輸出：
    {
      "sequence_analysis": "請先分析畫面如何呈現這 40 秒從靜到動的節奏。",
      "visual_features": "簡述光影與空間美學。",
      "ad_script": "最終生成的口述影像詞稿（120-150字，須包含刺殺結局與寫意美學）。"
    }
    """

    content_messages = [
        {"type": "text", "text": user_prompt_text}
    ]

    # 加上 detail: "low" 以防 Token 爆炸
    for img in image_frames:
        content_messages.append({
            "type": "image_url",
            "image_url": {
                "url": f"data:image/jpeg;base64,{img}",
                "detail": "low"
            }
        })

    response = client.chat.completions.create(
        model="gpt-4o",
        response_format={ "type": "json_object" },
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": content_messages}
        ],
        temperature=0.5
    )
    return response.choices[0].message.content

# 執行並印出結果
print("=== 華語 AD 測試結果  ===")
result_mandarin = generate_ad(system_prompt_mandarin, frames)
print(result_mandarin)

=== 華語 AD 測試結果  ===
{
  "sequence_analysis": "畫面從靜謐的樹林開始，隱娘如影般穿梭，營造出壓迫感。隨著男子騎馬經過，隱娘迅速逼近，伏擊瞬間爆發。結尾男子墜馬，畫面劇烈搖晃，從靜到動的節奏轉變鮮明。",
  "visual_features": "黑白光影交錯，樹林中隱娘的身影若隱若現，空間感狹窄而壓迫。光線透過樹葉灑下，營造出神秘而冷峻的氛圍。",
  "ad_script": "隱娘在斑駁的樹影間無聲移動，目光如炬。男子騎著華麗的馬匹緩緩前行，毫無察覺。隱娘如鬼魅般逼近，動作如閃電般迅捷，刀光一閃，男子從馬上墜落。畫面劇烈搖晃，模糊不清，結局在瞬間定格。靜謐的樹林再次回歸沉寂。"
}


### 華語準則 無提示

In [ ]:
def generate_ad(system_prompt, image_frames):

    user_prompt_text = """


    【執行與輸出要求】
在遵循上述準則草擬出描述後，請重新檢視這些準則，檢查是否有任何遺漏或違背之處，並修改描述以確保完全符合規範。

由於這些描述將作為影片播放時的連續旁白，描述聽起來必須像是一個連貫的敘事，而不是分割成獨立的畫面。請絕對不要在你的描述中提及「影格」、「在這張圖片中」或「畫面顯示」等字眼。

請將每段口述影像輸出於新的一行。每段口述影像應控制在 [請填入最大字數限制，例如：100] 個字左右。如果你因為某些原因無法提供描述，請明確說明原因。

    請以 JSON 格式輸出：
    {
      "sequence_analysis": "請先分析畫面如何呈現這 40 秒從靜到動的節奏。",
      "visual_features": "簡述光影與空間美學。",
      "ad_script": "最終生成的口述影像詞稿（120-150字，須包含刺殺結局與寫意美學）。"
    }
    """

    content_messages = [
        {"type": "text", "text": user_prompt_text}
    ]

    # 加上 detail: "low" 以防 Token 爆炸
    for img in image_frames:
        content_messages.append({
            "type": "image_url",
            "image_url": {
                "url": f"data:image/jpeg;base64,{img}",
                "detail": "low"
            }
        })

    response = client.chat.completions.create(
        model="gpt-4o",
        response_format={ "type": "json_object" },
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": content_messages}
        ],
        temperature=0.5
    )
    return response.choices[0].message.content

# 執行並印出結果
print("=== 華語 AD 測試結果  ===")
result_mandarin = generate_ad(system_prompt_mandarin, frames)
print(result_mandarin)

=== 華語 AD 測試結果  ===
{
  "sequence_analysis": "畫面從靜謐的森林中開始，人物隱藏在樹影間，營造出緊張的氛圍。隨著鏡頭轉向騎馬的目標，節奏逐漸加快。最後，刺客迅速出現，動作果斷，完成刺殺。",
  "visual_features": "黑白光影交錯，樹影斑駁，營造出神秘而壓抑的空間。人物在樹間移動，與背景融為一體，突顯出刺客的隱匿與決絕。",
  "ad_script": "樹林靜謐，樹影交錯，刺客隱匿其中。她凝視前方，目光如刃。騎馬者緩緩出現，衣袍隨風飄動。刺客如幽靈般掠出，匕首在手，目標近在咫尺。瞬間，她迅速出手，劍光一閃，目標應聲而倒。光影隨風而逝，樹林重歸寂靜，唯有殘影留存。"
}


### 英文準則

In [ ]:
# English 42 準則
system_prompt_english = """
你是一位頂尖的影像描述（口述影像，Audio Description）專家。你將觀看一段影片的數個關鍵影格（keyframes）。請針對這些連續影格提供專業的口述影像，並嚴格遵守以下所有準則。這些準則是為了協助視障（盲與低視力）使用者理解影片所制定的，你必須應用【所有】準則。

【口述影像 42 項核心準則】
1. 避免過度描述：不要包含非必要的視覺細節。
2. 保持客觀中立：除非內容特別要求，否則描述不應帶有主觀意見或評論。
3. 依劇情相關性取捨：根據場景與劇情的相關程度，決定細節描述的深淺。
4. 敘事視角與時態：描述應具資訊性且口語自然，使用「現在式」與「第三人稱全知視角」。
5. 詞彙精準度：詞彙應反映節目的主要語言環境，符合該影視類型的基調，並顧及目標受眾的理解力。確保用詞準確、清晰且簡潔。
6. 歷史與文化敏感度：考量歷史背景，避免使用帶有負面意涵或偏見的詞彙。
7. 動詞優先：選擇生動、具畫面感的動詞，避免使用平淡動詞搭配大量副詞。
8. 慎用代名詞：僅在指代對象絕對清晰無歧義時才使用代名詞。
9. 具象化比喻：描述形狀和大小時，請使用大眾熟悉且具全球共通性的物品進行比喻。
10. 保持一致性：在所有的描述中，對詞彙選擇、角色特質和視覺元素的稱呼必須保持一致。
11. 受眾適配性：語氣和詞彙必須符合目標受眾的年齡層。
12. 零錯誤產出：確保選詞、發音、措辭或咬字（生成文字的語意）沒有錯誤。
13. 敘事結構：先提供整體的環境脈絡，再逐步補充局部細節。
14. 物理特徵描述：根據內容適當描述物體的形狀、大小、紋理或顏色。
15. 第一人稱視角（例外情況）：若為了增強觀眾的沉浸感與參與感，可視情況使用第一人稱敘事。
16. 明確指示：適當使用冠詞（量詞或指示代名詞）來介紹或指稱主體。
17. 語域選擇：除非情境需要，否則優先使用正式用語而非口語/俚語。
18. 術語定義：引入新術語、物品或動作時，先標示其名稱，隨後再給予定義或解釋。
19. 絕對客觀與不審查：客觀描述，不加個人解釋或評論；絕對不可審查、過濾或刪減爭議內容。
20. 語氣基調：以平穩、客觀（但不單調）的語氣傳達，並與節目的整體基調保持一致。
21. 情感隨劇情起伏：在適當的時機加入情感、興奮感或輕鬆感。根據節目類型調整情緒和氛圍。
22. 兒童內容特化：若是兒童節目，請針對兒童的理解力調整語言與節奏。
23. 忠於視覺畫面：不要更改、過濾或排除內容。請「描述你所看到的」。在描述中追求簡單與簡潔。
24. 動作優先級：描述動作時，優先處理最相關的內容，以免干擾使用者的觀影體驗。
25. 交代時空背景：當地點、時間和天氣條件與場景或劇情相關時，必須將其納入描述。
26. 聚焦核心意圖：聚焦於具有學習與娛樂價值的關鍵內容，確保傳達節目的核心意圖。
27. 教學影片規範：若是教學或指導性影片，請優先描述活動的操作順序。
28. 戲劇作品規範：若是戲劇製作，請包含風格、背景設定、視覺焦點、時代、服裝、五官特徵、物品和美學等元素。
29. 學習成果導向：描述觀眾為了跟隨劇情、理解並欣賞影片預期學習成果所「最需要知道」的精華資訊。
30. 描述涵蓋範圍：口述影像必須涵蓋角色、地點、時間與環境、畫面上的動作，以及畫面上的資訊。
31. 僅限可見事物：僅描述明眼觀眾能直接看到的內容（不腦補未發生的事）。
32. 角色特徵描述：描述與身分和性格相關的主要及關鍵配角視覺特徵。優先對頭髮、皮膚、眼睛、體格、身高、年齡和明顯身心障礙等特徵進行「基於事實」的描述。確保一致性，避免針對特定特徵將角色單獨挑出，並使用「以人為本」（person-first）的語言。
33. 勿猜測身分認同：若無法確認或劇情尚未交代，請勿猜測或假設角色的種族、民族或性別認同。
34. 角色首次登場：首次提到角色名字時，盡量在名字前加上視覺特徵描述（例如：一名蓄鬍的男子，傑克）。
35. 表情與肢體：描述必須精準傳達角色的面部表情、肢體語言和反應。
36. 種族術語：當對內容含義/意圖很重要時，請使用目前社會廣泛接受的術語來描述種族。
37. 性別表現：避免僅以「性別表現」來識別角色，除非它能為視障觀眾提供其他方式無法察覺的獨特見解。
38. 服裝描述：若服裝能增強角色塑造、劇情推動、背景設定或類型觀賞性，請描述角色的服裝。
39. 畫面文字處理：若畫面上的文字對理解至關重要，請建立一套朗讀畫面文字的固定模式（例如：提示「畫面上出現文字：」）。
40. 字幕處理：遇到外語字幕時，描述者應在聲明「出現字幕」後，直接朗讀翻譯內容。
41. 鏡頭切換提示：當鏡頭切換對理解場景至關重要時，請透過描述「動作發生的地點」或「角色在新鏡頭中出現的位置」來暗示鏡頭的切換。
42. 提前播報：請在內容發生「之前」提供描述，而非發生之後。



請以 JSON 格式輸出：
{
  "visual_features": "簡述你提取到的關鍵視覺特徵與美學氛圍",
  "ad_script": "最終生成的口述影像詞稿（120-150字）"
}
"""

### 英文準則+提示

In [ ]:
def generate_ad(system_prompt, image_frames):
    user_prompt_text = """
    這是電影《刺客聶隱娘》的一段連續畫面（按時間先後順序排列，總長約 40 秒）。

    【人類專家劇情提示】
    注意這 40 秒內的節奏變化，這是一個「從靜到動」的過程：
    1. 前半段（靜）：隱娘在黑白斑駁的樹林中靜靜埋伏、穿梭。一位男子騎著裝飾華麗的馬匹緩緩經過。
    2. 後半段（動）：隱娘突然如鬼魅般逼近發動伏擊，動作極快。
    3. 結局：騎馬的男子被一擊斃命，從馬上墜落，畫面變得劇烈搖晃且模糊。

    【執行與輸出要求】
在遵循上述準則草擬出描述後，請重新檢視這些準則，檢查是否有任何遺漏或違背之處，並修改描述以確保完全符合規範。

由於這些描述將作為影片播放時的連續旁白，描述聽起來必須像是一個連貫的敘事，而不是分割成獨立的畫面。請絕對不要在你的描述中提及「影格」、「在這張圖片中」或「畫面顯示」等字眼。

請將每段口述影像輸出於新的一行。每段口述影像應控制在 [請填入最大字數限制，例如：100] 個字左右。如果你因為某些原因無法提供描述，請明確說明原因。

    請以 JSON 格式輸出：
    {
      "sequence_analysis": "請先分析畫面如何呈現這 40 秒從靜到動的節奏。",
      "visual_features": "簡述光影與空間美學。",
      "ad_script": "最終生成的口述影像詞稿（120-150字，須包含刺殺結局與寫意美學）。"
    }
    """

    content_messages = [
        {"type": "text", "text": user_prompt_text}
    ]

    # 加上 detail: "low" 以防 Token 爆炸
    for img in image_frames:
        content_messages.append({
            "type": "image_url",
            "image_url": {
                "url": f"data:image/jpeg;base64,{img}",
                "detail": "low"
            }
        })

    response = client.chat.completions.create(
        model="gpt-4o",
        response_format={ "type": "json_object" },
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": content_messages}
        ],
        temperature=0.5
    )
    return response.choices[0].message.content

# 執行並印出結果
print("=== ennglish 42 AD 測試結果  ===")
result_english = generate_ad(system_prompt_english, frames)
print(result_english)

=== ennglish 42 AD 測試結果  ===
{
  "sequence_analysis": "畫面從靜謐的樹林開始，隱娘在樹間緩步移動，營造出寧靜的氛圍。隨著男子騎馬進入視野，節奏逐漸加快。隱娘突然發動攻擊，動作迅捷，畫面轉為劇烈搖晃，男子從馬上墜落，整體節奏由靜轉動，形成強烈對比。",
  "visual_features": "黑白影像營造出古典且神秘的氛圍，樹林的光影斑駁增添了畫面的深邃感。空間上，樹木縱橫交錯，形成自然的屏障，隱娘在其中若隱若現，增強了隱秘的感覺。",
  "ad_script": "在黑白交錯的樹林中，隱娘靜靜地埋伏著，樹影斑駁，增添了神秘感。一位男子騎著華麗的馬匹緩緩經過，隱娘如鬼魅般迅速逼近，動作如閃電般迅捷。男子驚愕之際，被一擊斃命，從馬上墜落。畫面劇烈搖晃，模糊中透露出一絲悲劇的美感。"
}


### 英文準則+無提示

In [ ]:
def generate_ad(system_prompt, image_frames):
    user_prompt_text = """


    【執行與輸出要求】
在遵循上述準則草擬出描述後，請重新檢視這些準則，檢查是否有任何遺漏或違背之處，並修改描述以確保完全符合規範。

由於這些描述將作為影片播放時的連續旁白，描述聽起來必須像是一個連貫的敘事，而不是分割成獨立的畫面。請絕對不要在你的描述中提及「影格」、「在這張圖片中」或「畫面顯示」等字眼。

請將每段口述影像輸出於新的一行。每段口述影像應控制在 [請填入最大字數限制，例如：100] 個字左右。如果你因為某些原因無法提供描述，請明確說明原因。

    請以 JSON 格式輸出：
    {
      "sequence_analysis": "請先分析畫面如何呈現這 40 秒從靜到動的節奏。",
      "visual_features": "簡述光影與空間美學。",
      "ad_script": "最終生成的口述影像詞稿（120-150字，須包含刺殺結局與寫意美學）。"
    }
    """

    content_messages = [
        {"type": "text", "text": user_prompt_text}
    ]

    # 加上 detail: "low" 以防 Token 爆炸
    for img in image_frames:
        content_messages.append({
            "type": "image_url",
            "image_url": {
                "url": f"data:image/jpeg;base64,{img}",
                "detail": "low"
            }
        })

    response = client.chat.completions.create(
        model="gpt-4o",
        response_format={ "type": "json_object" },
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": content_messages}
        ],
        temperature=0.5
    )
    return response.choices[0].message.content

# 執行並印出結果
print("=== ennglish 42 AD 測試結果  ===")
result_english = generate_ad(system_prompt_english, frames)
print(result_english)

=== ennglish 42 AD 測試結果  ===
{
  "sequence_analysis": "影片從靜止的森林場景開始，光影交錯，樹木形成自然的屏障。隨著時間推移，角色逐漸出現，動作緩慢且充滿張力。最後，場景轉為快速動作，突顯刺殺的緊張氛圍。",
  "visual_features": "影片運用黑白對比強烈的光影效果，強調空間的深邃與神秘感。樹木的陰影與人物的輪廓交織，營造出一種詩意的氛圍。",
  "ad_script": "在靜謐的森林中，光影錯落，樹木如天然屏障。隱藏其中的身影逐漸顯現，步伐緩慢而堅定。隨著鏡頭轉向，一位騎士在樹間穿行，劍光閃爍。突然，鏡頭切換，刺客迅速接近，動作果斷，刀光劃過，結束了騎士的生命。整體畫面以黑白色調呈現，強調了刺殺場景的緊張與詩意美學。"
}
